# Exercise 2: Collecting data from the web

In exercise 1, the files were already on your computer. Now, we will **get them directly from the web**, in two ways:
- by downloading a **web page** (HTML) and extracting information from it: this is **web scraping**,
- by asking a service for data through its **API**, which answers in JSON.

| Part | What we do | Tool |
|---|---|---|
| 2a | Download the pages of two movies on [TMDB](https://www.themoviedb.org) | `requests` |
| 2b | Get tomorrow's weather forecast on the FUCaM campus | `requests` + the Open-Meteo API |
| 2c | Extract the synopsis and the actors of the two movies | `BeautifulSoup` |
| Going further | Scrape more movies and save the data for the rest of the course | files |

### How to work

- Run the cells **one by one, from top to bottom** (`Shift + Enter`).
- ✏️ marks a cell you have to complete: replace the lines marked `## @COMPLETE`.
- 🧪 marks a test cell: run it to check your answer before moving on.
- 💡 gives you a hint, 🤔 asks you a question to think about.

In [ ]:
import os

import requests
from bs4 import BeautifulSoup

## Exercise 2a: Download web pages

**Goal**: download automatically the TMDB pages of *Harry Potter 1* and *The Lord of the Rings 1*.

When your browser opens a page, it sends an **HTTP request** to a server, which answers with the **HTML code** of the page. With Python, the `requests` library does the same thing:

```python
response = requests.get(url)   # send the request
response.status_code           # 200 means "OK", 404 means "page not found", ...
response.text                  # the HTML code of the page, as a string
```

Each movie on TMDB has an identifier, visible at the end of its URL:

| Movie | Identifier | Page |
|---|---|---|
| Harry Potter and the Philosopher's Stone | `671` | https://www.themoviedb.org/movie/671 |
| The Lord of the Rings: The Fellowship of the Ring | `120` | https://www.themoviedb.org/movie/120 |

We add `?language=en-US` at the end of the URL to get the page in English, whatever your location.

✏️ **Complete the function `download_html`.**

In [ ]:
BASE_URL = "https://www.themoviedb.org/movie/"


def download_html(url):
    """Download a web page and return its HTML code as a string."""
    ## @COMPLETE : send a GET request to the url and store the answer in `response`
    response = None

    ## @COMPLETE : check that the request succeeded (status code 200), print a message otherwise
    ...

    ## @COMPLETE : return the HTML content of the page (as text)
    return None

🧪 **Test**: you should see a long string starting with `<!DOCTYPE html>`.

In [ ]:
html = download_html(BASE_URL + "671" + "?language=en-US")

print("Number of characters:", len(html))
print(html[:300])

Now, let's download both movies. The pages are stored in a dictionary: `movie title -> HTML`.

In [ ]:
movie_ids = {
    "Harry Potter 1": "671",
    "The Lord of the Rings 1": "120",
}

pages = {}
for title, movie_id in movie_ids.items():
    pages[title] = download_html(BASE_URL + movie_id + "?language=en-US")
    print(title, ":", len(pages[title]), "characters downloaded")

## Exercise 2b: Use an API

**Goal**: use `requests` and the API of [open-meteo.com](https://open-meteo.com) to get **tomorrow's weather forecast on the FUCaM campus** (Mons).

A web page is made for humans: the information is hidden in the HTML and we must dig it out. An **API** (*Application Programming Interface*) is made for programs: we ask a precise question in the URL and the server answers with **clean data in JSON**, the format you loaded in exercise 1.

To ask a question to Open-Meteo, we send a request to `https://api.open-meteo.com/v1/forecast` with some **parameters**. `requests` builds the URL for us from a dictionary:

```python
response = requests.get(url, params={"latitude": 50.45, "longitude": 3.95})
# -> https://api.open-meteo.com/v1/forecast?latitude=50.45&longitude=3.95
data = response.json()   # converts the JSON answer into a Python dictionary
```

The available parameters are described in the [documentation](https://open-meteo.com/en/docs). We use:
- `latitude`, `longitude`: the location of the campus,
- `daily`: the daily values we want (minimum and maximum temperature, total precipitation),
- `timezone`: so that "a day" means a day in Belgium,
- `forecast_days`: `2` means today and tomorrow.

✏️ **Send the request and convert the answer into a dictionary called `forecast`.**

In [ ]:
API_URL = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 50.45,   # FUCaM campus, Mons
    "longitude": 3.95,
    "daily": "temperature_2m_min,temperature_2m_max,precipitation_sum",
    "timezone": "Europe/Brussels",
    "forecast_days": 2,
}

## @COMPLETE : send a GET request to API_URL with the parameters
response = None

## @COMPLETE : convert the JSON answer into a dictionary
forecast = None

🧪 **Test**: look at the structure of the answer. The forecast is in `forecast["daily"]`: each key contains a **list with one value per day** (index `0` is today, index `1` is tomorrow).

In [ ]:
forecast

✏️ **Print tomorrow's date, minimum and maximum temperature, and precipitation.**

💡 *Hint*: `forecast["daily"]["temperature_2m_max"][1]` is tomorrow's maximum temperature.

In [ ]:
daily = forecast["daily"]

## @COMPLETE : print tomorrow's forecast

🤔 **Question**: should you take an umbrella tomorrow? Compare with the forecast of your favourite weather app.

## Exercise 2c: Extract the synopsis and the actors

**Goal**: from the HTML pages downloaded in 2a, extract the **synopsis** and the **list of actors** of each movie.

### Parse the HTML

As in exercise 1, BeautifulSoup turns the HTML into a tree of tags that we can search. We will use two methods:
- `soup.find("div", class_="xxx")`: returns the **first** `<div>` tag whose class is `xxx`,
- `soup.select("css selector")`: returns the **list** of all tags matching a CSS selector.

On any tag, `.text` gives the text it contains.

In [ ]:
soup = BeautifulSoup(pages["Harry Potter 1"], "html.parser")

print(soup.title.text)

### The synopsis

To know where the information is located in the page, use your browser:
1. open https://www.themoviedb.org/movie/671,
2. right-click on the synopsis (below "Overview") and choose **Inspect**,
3. look at the HTML around it.

You should find something like this:

```html
<div class="overview" dir="auto">
  <p>Harry Potter has lived under the stairs at his aunt and uncle's house ...</p>
</div>
```

So the synopsis is the text of the `<div>` whose class is `overview`.

✏️ **Complete the function `extract_synopsis`.**

💡 *Hint*: `class` is a reserved word in Python, which is why BeautifulSoup uses `class_` (with an underscore). The string method `.strip()` removes the spaces and line breaks at the beginning and the end of a string.

In [ ]:
def extract_synopsis(soup):
    """Return the synopsis of the movie."""
    ## @COMPLETE : find the <div> whose class is "overview"
    overview_div = None

    ## @COMPLETE : return its text, without the spaces and line breaks around it
    return None

🧪 **Test**: you should see the synopsis of Harry Potter.

In [ ]:
print(extract_synopsis(soup))

### The actors

Inspect the "Top Billed Cast" section of the page. Each actor is displayed on a card:

```html
<ol class="people scroller">
  <li class="card">
    ...
    <p><a href="/person/10980-daniel-radcliffe">Daniel Radcliffe</a></p>
    <p class="character">Harry Potter</p>
  </li>
  ...
</ol>
```

The CSS selector `"ol.people.scroller li.card p a"` reads as: *the `<a>` links, inside a `<p>`, inside a `<li class="card">`, inside the `<ol class="people scroller">`*. It is already written for you: `soup.select(...)` returns the **list** of these links.

✏️ **Complete the function `extract_actors`** so that it returns the list of actor **names** (strings), not the list of tags.

💡 *Hint*: loop over `actor_links` and take the `.text` of each link (a list comprehension works well here).

In [ ]:
def extract_actors(soup):
    """Return the list of actors (names) displayed on the movie page."""
    actor_links = soup.select("ol.people.scroller li.card p a")

    ## @COMPLETE : build the list of actor names (the text of each link)
    actors = []

    return actors

🧪 **Test**: you should see a list of names starting with `'Daniel Radcliffe', 'Rupert Grint', 'Emma Watson'`.

In [ ]:
print(extract_actors(soup))

### Both movies

Our two functions work on any TMDB page. Let's apply them to both movies:

In [ ]:
for title, html in pages.items():
    soup = BeautifulSoup(html, "html.parser")
    print("=====", title)
    print(extract_synopsis(soup))
    print(extract_actors(soup))
    print()

## Going further: build the dataset for the rest of the course

We now have all the building blocks of a small **scraper**. In this last part, nothing to complete: run the cells and make sure you understand each line. We will:
1. scrape a few more movies,
2. save one text document per movie, used in the **text mining** part of the course,
3. build a network of actors, used in the **link analysis** part of the course.

The function `scrape_movie` chains everything: download → parse → extract. It returns a **dictionary** describing the movie.

In [ ]:
def scrape_movie(movie_id):
    """Download the page of a movie and return its synopsis and actors."""
    html = download_html(BASE_URL + movie_id + "?language=en-US")
    soup = BeautifulSoup(html, "html.parser")
    return {
        "synopsis": extract_synopsis(soup),
        "actors": extract_actors(soup),
    }

These are the four movies analysed in the **text mining** exercises. You can add other movies later, but run the notebook once with these four first.

In [ ]:
movies_to_scrape = {
    "Harry Potter 1": "671",
    "Harry Potter 2": "672",
    "The Lord of the Rings 1": "120",
    "The Hobbit 1": "49051",
}

movies = []
for title, movie_id in movies_to_scrape.items():
    print("Scraping", title, "...")
    movies.append(scrape_movie(movie_id))

print(len(movies), "movies scraped")

### Save one document per movie

Each file `docs/doc_<i>.txt` contains the title of a movie (first line), its synopsis, then its actors (one per line).

In [ ]:
OUTPUT_DIR = "docs"
os.makedirs(OUTPUT_DIR, exist_ok=True)  # create the folder if it does not exist

for i, (title, movie) in enumerate(zip(movies_to_scrape, movies), start=1):
    path = os.path.join(OUTPUT_DIR, f"doc_{i}.txt")
    with open(path, "w", encoding="utf-8") as file:
        file.write(title + "\n")
        file.write(movie["synopsis"] + "\n")
        for actor in movie["actors"]:
            file.write(actor + "\n")
    print("Saved", path)

### Build the network of actors

We build a network where:
- each **node** is an actor,
- two actors are **linked** if they played in the same movie,
- the **weight** of a link is the number of movies they played in together.

First, we give each actor a unique identifier (1, 2, 3, ...):

In [ ]:
actor_ids = {}  # actor name -> identifier
for movie in movies:
    for actor in movie["actors"]:
        if actor not in actor_ids:
            actor_ids[actor] = len(actor_ids) + 1

print(len(actor_ids), "different actors")

Then, for each movie, we look at **every pair of actors** in its cast and add 1 to the weight of their link.

The weights are stored in a dictionary whose keys are pairs `(id1, id2)`, always with `id1 < id2` so that the pair (A, B) and the pair (B, A) are counted as the same link.

In [ ]:
links = {}  # (id1, id2) -> number of movies played together
for movie in movies:
    ids = [actor_ids[actor] for actor in movie["actors"]]
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            pair = (min(ids[i], ids[j]), max(ids[i], ids[j]))
            links[pair] = links.get(pair, 0) + 1

print(len(links), "links")

🤔 **Question**: the actors of a movie are all linked to each other. But is our network made of separate groups, one per movie, or are some movies connected? Run the cell below to find the actors who play in more than one movie: they are the "bridges" between movies. Which links have a weight of 2?

In [ ]:
for actor in actor_ids:
    titles = [title for title, movie in zip(movies_to_scrape, movies) if actor in movie["actors"]]
    if len(titles) > 1:
        print(actor, "plays in", titles)

Finally, we save the network in two CSV files:
- `actors.txt`: one line per actor, `identifier,"name"`,
- `links.txt`: one line per link, `id1,id2,weight`.

In [ ]:
with open(os.path.join(OUTPUT_DIR, "actors.txt"), "w", encoding="utf-8") as file:
    for actor, actor_id in actor_ids.items():
        file.write(f'{actor_id},"{actor}"\n')

with open(os.path.join(OUTPUT_DIR, "links.txt"), "w", encoding="utf-8") as file:
    for (id1, id2), weight in links.items():
        file.write(f"{id1},{id2},{weight}\n")

print("Network saved in", OUTPUT_DIR)

## 🎉 Well done!

You can now collect data from the web in two ways:

| | Web scraping | API |
|---|---|---|
| Made for | humans | programs |
| Answer | an HTML page | clean data (JSON) |
| Tools | `requests` + `BeautifulSoup` | `requests` + `.json()` |
| Difficulty | find the right tags with *Inspect* | read the documentation |

⚠️ Websites change their HTML regularly: a scraper that works today may break tomorrow. If a step suddenly fails, inspect the page again and adapt the tag or the class you are looking for. When a website offers an API, prefer it.